<div dir="rtl">

# تمرین پایانی ۱۳
# از شمارش کلمات تا ترنسفورمر

این نوت‌بوک **مسیر کار** است، نه راه‌حل. صورت کامل سؤال‌ها در فایل PDF است:

`week13_hw_projects.pdf`

در هر بخش:
1. اول markdown همان بخش را بخوانید.
2. سلول کد را خودتان پر کنید.
3. توضیح کوتاه را در سلول «توضیح این بخش» بنویسید.

کد آماده را از کلاس کپی نکنید. فقط ایده بگیرید.


<style>
.jp-MarkdownCell .jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.text_cell_render,
.rendered-markdown,
[dir="rtl"] {
  direction: rtl;
  text-align: right;
}
[dir="rtl"] p,
[dir="rtl"] li,
[dir="rtl"] ul,
[dir="rtl"] ol,
[dir="rtl"] h1,
[dir="rtl"] h2,
[dir="rtl"] h3,
[dir="rtl"] h4,
[dir="rtl"] blockquote,
[dir="rtl"] table {
  text-align: right;
}
[dir="rtl"] code {
  direction: ltr;
  unicode-bidi: isolate;
}
[dir="rtl"] pre,
[dir="rtl"] pre code {
  direction: ltr;
  unicode-bidi: isolate;
  text-align: left;
}
[dir="rtl"] blockquote {
  border-left: none;
  border-right: 4px solid #d0d7de;
  padding-left: 0;
  padding-right: 1em;
}
[dir="ltr"] {
  unicode-bidi: isolate;
}
</style>

</div>


<div dir="rtl">

## آماده‌سازی

</div>


In [1]:
# اگر روی Colab هستید و بسته‌ای کم دارید، این خط را از حالت توضیح خارج کنید.
# %pip install torch scikit-learn transformers -q

import math
import re
from collections import Counter
import numpy as np

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import torch
from torch import nn

torch.manual_seed(0)


<div dir="rtl">

# پروژه ۱. خبرخوان کوچک: از شمارش کلمات تا Encoder

**هدف:** یک طبقه‌بند کوچک برای دو موضوع خبری بسازید و ببینید BoW، embedding و Encoder هر کدام چه اطلاعاتی می‌بینند.

**داده:** `20 Newsgroups` فقط دو کلاس، حجم عمداً کوچک.

</div>


<div dir="rtl">

## بخش ۱ـ الف. داده، تمیزکاری و نگاه اول

باید انجام دهید:
- دو کلاس `sci.space` و `rec.sport.hockey` را بارگذاری کنید.
- حداکثر ۲۵۰ سند آموزش و ۱۰۰ سند آزمون بردارید.
- تابع نرمال‌سازی بنویسید.
- تعداد نمونه، میانگین طول، و دو نمونهٔ خام در کنار نسخهٔ تمیز را چاپ کنید.

</div>


In [2]:
CATEGORIES = ["sci.space", "rec.sport.hockey"]
MAX_TRAIN = 250
MAX_TEST = 100


def normalize_text(text: str) -> str:
    ### START CODE HERE ###
    text = re.sub(r'[^a-zA-Z\s]', '', text)

     # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Convert to lowercase
    text = text.lower()
    # lowercase, remove URLs, drop extra characters, collapse spaces
    return text
    ### END CODE HERE ###


### START CODE HERE ###
# 1) load 20 Newsgroups with sklearn (only CATEGORIES)
train_set = fetch_20newsgroups(
    subset="train",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes")
)

test_set = fetch_20newsgroups(
    subset="test",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes")
)


In [3]:
# 2) keep at most MAX_TRAIN / MAX_TEST documents
train_set.data = train_set.data[:MAX_TRAIN]
train_set.target = train_set.target[:MAX_TRAIN]

test_set.data = test_set.data[:MAX_TEST]
test_set.target = test_set.target[:MAX_TEST]
# 3) optionally remove headers/footers/quotes
# 4) apply normalize_text
train_texts, train_labels = [normalize_text(data) for data in train_set.data], train_set.target                    
test_texts, test_labels = [normalize_text(data) for data in test_set.data], test_set.target
### END CODE HERE ###

print("train size:", len(train_texts))
print("test size:", len(test_texts))
print("class counts:", Counter(train_labels))
print("mean train length (words):", np.mean([len(i) for i in train_texts]))
print("RAW :", train_set.data[:2])
print("CLEAN:", train_texts[:2])


train size: 250
test size: 100
class counts: Counter({np.int64(0): 126, np.int64(1): 124})
mean train length (words): 1257.696
RAW : ["Individual leaders by total points (Final standings)\n    NOTE: Games played and points per games not accurate !!\n\n    Player       Team   GP  G   A  Pts ppg  Prj PIM +/-\n\n    M.Lemieux    PIT    59  69  91 160 2.71 160  38  53\n    LaFontaine   BUF    82  53  95 148 1.80 148  63  13\n    Oates        BOS    83  45  97 142 1.71 142  32  12\n    Yzerman      DET    83  58  79 137 1.65 137  44  33\n    Turgeon      NYI    80  58  74 132 1.65 132  26  -2\n    Selanne      WIN    82  76  56 132 1.61 132  45   6\n    Mogilny      BUF    75  76  51 127 1.69 127  40   9\n    Gilmour      TOR    81  32  95 127 1.57 127  96  32\n    Robitaille   LA     82  63  62 125 1.52 127 100  16\n    Recchi       PHI    81  53  70 123 1.52 123  95  -2\n    Sundin       QUE    79  47  67 114 1.44 114  96  19\n    Stevens      PIT    71  55  57 112 1.58 112 169  16\n    B

<div dir="rtl">

### توضیح بخش ۱ـ الف

- تعداد نمونه در هر کلاس:
- میانگین طول:
- مشاهدهٔ شما از خام در برابر تمیز:

</div>


<div dir="rtl">

## بخش ۱ـ ب. خط پایهٔ شمارش کلمات

باید انجام دهید:
- BoW یا TF-IDF
- یک مدل خطی مثل Logistic Regression
- دقت آموزش و آزمون
- دو سند اشتباه
- دو جملهٔ خودتان که زیر BoW یکی می‌شوند

</div>


In [4]:
### START CODE HERE ###
# vectorize clean train/test texts
# fit a linear classifier
# print train accuracy and test accuracy
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(train_texts)
X_test_bow = bow_vectorizer.transform(test_texts)

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_test_tfidf = tfidf_vectorizer.transform(test_texts)

bow_model = LogisticRegression()
bow_model.fit(X_train_bow, train_labels)

tfidf_model = LogisticRegression()
tfidf_model.fit(X_train_tfidf, train_labels)

bow_train_acc = bow_model.score(X_train_bow, train_labels)
bow_test_acc = bow_model.score(X_test_bow, test_labels)

tfidf_train_acc = tfidf_model.score(X_train_tfidf, train_labels)
tfidf_test_acc = tfidf_model.score(X_test_tfidf, test_labels)
### END CODE HERE ###

print("BoW train acc:", bow_train_acc)
print("BoW test acc :", bow_test_acc)


print("TfIdf train acc:", tfidf_train_acc)
print("TfIdf test acc :", tfidf_test_acc)

BoW train acc: 1.0
BoW test acc : 0.85
TfIdf train acc: 1.0
TfIdf test acc : 0.92


In [5]:
### START CODE HERE ###
# find two test documents the BoW model gets wrong

bow_preds = bow_model.predict(X_test_bow)
wrong_predicted_data_bow1 = test_texts[[int(i) for i in list(np.where(bow_preds != test_labels)[0])][0]]
wrong_predicted_data_bow2 = test_texts[[int(i) for i in list(np.where(bow_preds != test_labels)[0])][1]]
wrong_predicted_labels_bow = test_labels[bow_preds != test_labels]
wrong_predictions_bow = bow_preds[bow_preds != test_labels]
print("wrong_predicted_data_bow1 ###: \n", wrong_predicted_data_bow1)
print("wrong_predicted_data_bow2 ###: \n", wrong_predicted_data_bow2)

print("wrong_predicted_labels_bow", wrong_predicted_labels_bow[:2])
print("wrong_predictions_bow", wrong_predictions_bow[:2])
print("\n\n")

tfidf_preds = tfidf_model.predict(X_test_tfidf)
wrong_predicted_data_tfidf1 = test_texts[[int(i) for i in list(np.where(tfidf_preds != test_labels)[0])][0]]
wrong_predicted_data_tfidf2 = test_texts[[int(i) for i in list(np.where(tfidf_preds != test_labels)[0])][1]]
wrong_predicted_labels_tfidf = test_labels[tfidf_preds != test_labels]
wrong_predictions_tfidf = tfidf_preds[tfidf_preds != test_labels]
print("wrong_predicted_data_tfidf1: ##\n", wrong_predicted_data_tfidf1)
print("wrong_predicted_data_tfidf2: ##\n")
print("wrong_predicted_labels_tfidf: ", wrong_predicted_labels_tfidf[:2])
print("wrong_predictions_tfidf: ", wrong_predictions_tfidf[:2])

# print a short slice of each and the true/predicted labels
### END CODE HERE ###

wrong_predicted_data_bow1 ###: 
 speaking of which a paper was out a few years ago about a weather sat imaging a lunar eclipse are those images uploaded anywhere i could dig out the reference if theres interest shag
wrong_predicted_data_bow2 ###: 
 bowman is in the hockey hall of fame as a coach alreadyand unlike gil steinhe didnt get their by stacking the board of directors bowman is the geniusjohnson isnt bowman is a great coachjohnson a very good one however bowman is really not a modern coach and patricks solution of having him only run the team on game days and delegate most of the daytoday responsibity to the assistants is a shrewd managerial decision on patricks part
wrong_predicted_labels_bow [1 0]
wrong_predictions_bow [0 1]



wrong_predicted_data_tfidf1: ##
 alsoif they did come from the oort cloud we would expect to see the same from other stars oort clouds
wrong_predicted_data_tfidf2: ##

wrong_predicted_labels_tfidf:  [1 1]
wrong_predictions_tfidf:  [0 0]


In [6]:
my_pair = [
    "Dog bites bone",
    "Bone bites dog",
]
### START CODE HERE ###
my_pair_normalized = [normalize_text(text) for text in my_pair]
bow = bow_vectorizer.fit_transform(my_pair_normalized)
# Vocabulary
print("Vocabulary:")
print(bow_vectorizer.get_feature_names_out())

# Dense BoW matrix
print("\nBoW:")
print(bow.toarray())
# show that these two sentences get the same BoW vector
### END CODE HERE ###

Vocabulary:
['bites' 'bone' 'dog']

BoW:
[[1 1 1]
 [1 1 1]]


<div dir="rtl">

### توضیح بخش ۱ ـ ب

- دو خطای مدل و حدس شما:
- 

در جمله دوم کلمات روز و هر روز تکرار شده ممکن است مدل آن را به آب و هوا نسبت دهد. در جمله اول هم کلمات نشستن و تصویر هم هست و ممکن است در جملات مرتبط با هاکی این کلمات هم تکرار شوند. در هر دو حالت جایگاهها اهمیت ندارند و حافظه ای هم نداریم.
- این دو جمله چرا زیر BoW یکی می‌شوند؟
- چون حساس به ترتیب کلمات نیست. صرفا فرکانس تکرار کلمات را محاسبه می‌کند. اینجا هم دو جمله کلمات دقیقا یکسان ولی با ترتیب متفاوتی دارند.
- BoW چه چیزی را می‌بیند و چه چیزی را نمی‌بیند؟
- کلمات یا به عبارتی توکنها را می‌بیند و دفعات تکرار آنها را می‌شمارد ولی نمی‌داند کلمات در جمله چه جایگاهی دارند یا ترتیب آنها در جمله کجا بوده. در زبان ترتیب خیلی مهم است. اینکه صفت و اسم وفعل چیستند و در کجا هستند میتوانند به معنی جمله اثر بگذارند یا ترتیب اشتباه آنها باعث میشوند جمله غلط و بی معنی باشد.

</div>


<div dir="rtl">

## بخش ۱ـ ج. از شناسهٔ کلمه به embedding

باید انجام دهید:
- واژه‌نامه از دادهٔ آموزش؛ کلمات نادر → `unk`
- هر سند → دنبالهٔ ID با طول ثابت
- `nn.Embedding` + میانگین توکن‌ها + لایهٔ خطی ۲ کلاسه
- چند epoch آموزش و مقایسه با BoW
- حدود ۱۰ جفت skip-gram با پنجرهٔ ۲

</div>


In [7]:
MAX_LEN = 64
MIN_FREQ = 2
UNK = "<unk>"
PAD = "<pad>"

# Build vocabulary from train_texts
vocab = {token for sentence in train_texts for token in sentence.split()}

stoi = {
    PAD: 0,
    UNK: 1,
}

stoi.update({
    token: idx
    for idx, token in enumerate(sorted(vocab), start=2)
})

itos = {idx: token for token, idx in stoi.items()}


def encode_documents(text):
    tokens = text.split()

    # Convert unknown words to <unk>
    tokenized = [
        stoi.get(token, stoi[UNK])
        for token in tokens
    ]

    # Truncate
    tokenized = tokenized[:MAX_LEN]

    # Pad
    tokenized.extend(
        [stoi[PAD]] * (MAX_LEN - len(tokenized))
    )

    return torch.tensor(tokenized, dtype=torch.long)


# Encode documents
X_train_ids = torch.stack([
    encode_documents(text)
    for text in train_texts
])

X_test_ids = torch.stack([
    encode_documents(text)
    for text in test_texts
])

# Labels
y_train = torch.tensor(train_labels, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)


### END CODE HERE ###

print("vocab size:", len(vocab))
print("X_train_ids:", X_train_ids[:1])


vocab size: 9458
X_train_ids: tensor([[4026, 4645, 1139, 8596, 6337, 2989, 7967, 5677, 3251, 6294,  330, 6337,
         6137, 3251, 5674,   60, 6295, 8336, 3438, 3230,    2, 6592, 6407, 6493,
         6252, 5300, 6261, 4557, 1088, 5728,  948, 9419, 2146, 8717, 5716, 7455,
         9249, 5332, 1088, 3357, 8588, 7107, 4546, 6805, 6208, 8190, 6663, 8044,
         6261, 1112, 8927, 8543, 6261, 7128, 1376, 4232, 8063, 7266, 6663, 4348,
          948, 3859, 8063,  346]])


In [8]:
from torch.utils.data import TensorDataset, DataLoader

train_set = TensorDataset(X_train_ids, y_train)
test_set = TensorDataset(X_test_ids, y_test)

train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_set,
    batch_size=32,
    shuffle=False
)

In [9]:
import torchmetrics

def evaluate_tm(model, data_loader, metric, device="cpu"):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, n_epochs, device="cpu"):
    history = {"train_losses": [], "train_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}")
    return history

In [10]:
class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_classes=2, pad_id=0):
        super().__init__()
        ### START CODE HERE ###
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.output = nn.Linear(d_model, 2)
        ### END CODE HERE ###

    def forward(self, token_ids):
        ### START CODE HERE ###
        embedding = self.embed(token_ids)
        return self.output(embedding.mean(dim=1))
        ### END CODE HERE ###


### START CODE HERE ###
# train a few epochs with CrossEntropyLoss
torch.manual_seed(42)
criterion = nn.CrossEntropyLoss()
model = MeanEmbeddingClassifier(len(vocab))
optimizer = torch.optim.Adam(model.parameters())
EPOCHS = 50
metric = torchmetrics.Accuracy(task="multiclass", num_classes=2)

history = train(model, optimizer, criterion, metric, train_loader, EPOCHS)

# print test accuracy
emb_test_acc = evaluate_tm(model, test_loader, metric).item()
### END CODE HERE ###

print("embedding test acc:", emb_test_acc)


Epoch 1/50, train loss: 0.7021, train metric: 0.4800
Epoch 2/50, train loss: 0.6973, train metric: 0.4960
Epoch 3/50, train loss: 0.6935, train metric: 0.5080
Epoch 4/50, train loss: 0.6892, train metric: 0.5160
Epoch 5/50, train loss: 0.6856, train metric: 0.5440
Epoch 6/50, train loss: 0.6809, train metric: 0.5600
Epoch 7/50, train loss: 0.6765, train metric: 0.5880
Epoch 8/50, train loss: 0.6717, train metric: 0.6200
Epoch 9/50, train loss: 0.6669, train metric: 0.6520
Epoch 10/50, train loss: 0.6616, train metric: 0.6720
Epoch 11/50, train loss: 0.6565, train metric: 0.7040
Epoch 12/50, train loss: 0.6504, train metric: 0.7360
Epoch 13/50, train loss: 0.6432, train metric: 0.7480
Epoch 14/50, train loss: 0.6358, train metric: 0.7840
Epoch 15/50, train loss: 0.6272, train metric: 0.7920
Epoch 16/50, train loss: 0.6201, train metric: 0.8280
Epoch 17/50, train loss: 0.6110, train metric: 0.8480
Epoch 18/50, train loss: 0.6011, train metric: 0.8560
Epoch 19/50, train loss: 0.5901, trai

In [11]:
def extract_grams(text, window=2):
    tokens = text.split()
    skipgram_pairs = []

    for i, center in enumerate(tokens):
        for offset in range(1, window + 1):
            if i - offset >= 0:
                skipgram_pairs.append((center, tokens[i - offset]))

            if i + offset < len(tokens):
                skipgram_pairs.append((center, tokens[i + offset]))

    return skipgram_pairs


skipgram_pairs = []

for text in train_texts:
    skipgram_pairs.extend(extract_grams(text, window=2))

print("skip-gram pairs:")
for center, context in skipgram_pairs[:10]:
    print(center, "->", context)

skip-gram pairs:
individual -> leaders
individual -> by
leaders -> individual
leaders -> by
leaders -> total
by -> leaders
by -> total
by -> individual
by -> points
total -> by


<div dir="rtl">

### توضیح بخش ۱ـ ج

- embedding نسبت به BoW چه چیزی اضافه کرد؟
- داده به فضای جدیدی برده میشود. در این فضا کلمات مشابه کنار هم قرار میگیرند و نامشابه از هم دور میشوند. به نوعی میشود گفت این لایه داده ها را به وکتورهای با معنی تبدیل میکند که جای مشخصی از فضای جدید را میگیرند ولی اعداد به دست آمده از متد قبلی این توانایی را ندارد و هر کلمه رفتار یکسانی با هر کلمه در کیسه را دارد.
- این جفت‌های skip-gram چه سیگنال آموزشی‌ای می‌دهند؟
- کلمات نزدیک به هم را استخراج میکند و سعی میکند به نوعی معنی هر کلمه را وابسته به کانتکتسی که در آن قرار میگیرد را کشف کند.

- چه چیزی هنوز کم است؟ (ترتیب / بافت جمله)
یک مورد محدودیت پنجره است. از یادگیری وابستگیهای به دور از پنجره عاجز است. از  طرفی جایگاه هر کلمه در جمله را نیز هنوز نمیداند. اینکه فعل است یا فاعل یا ...
و اینکه هر همسایگی چقدر وزن دارد.
</div>


<div dir="rtl">

## بخش ۱ـ د. Encoder کوچک

باید انجام دهید:
- embedding توکن + embedding موقعیت + یک یا دو لایه `TransformerEncoder`
- میانگین روی طول دنباله → لایهٔ کلاس
- **ماسک علّی نگذارید**
- شکل تنسور یک سند نمونه را چاپ کنید
- دقت آزمون را کنار مدل‌های قبلی بگذارید

</div>


In [12]:
class TinyNewsEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, num_layers=2, num_classes=2, max_len=64, pad_id=0):
        super().__init__()
        ### START CODE HERE ###
        self.embed = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=pad_id
        )
        self.pos_embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_id
        )
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)
        self.output = nn.Linear(d_model, num_classes)
        self.pad_id = pad_id
        ### END CODE HERE ###

    def forward(self, token_ids,print_shapes=False):
        ### START CODE HERE ###
        B, T = token_ids.shape
        if print_shapes:
            print(f"token_ids.shape: {token_ids.shape}\n")
        x = self.embed(token_ids)
        if print_shapes:
            print(f"token_encoding.shape: {x.shape}\n")
        positions = torch.arange(
            T,
            device=token_ids.device
        )

        position_embeddings = self.pos_embed(positions)
        if print_shapes:
            print(f"position_embeddings.shape: {position_embeddings.shape}\n")
        x = x + position_embeddings
        padding_mask = token_ids == self.pad_id
        x = self.encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        valid = (~padding_mask).unsqueeze(-1)
        x = (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)
        output = self.output(x)
        if print_shapes:
            print(f"output.shape: {output.shape}\n")
        return output
        ### END CODE HERE ###


### START CODE HERE ###
# instantiate, pass one batch, print shapes:
# token ids, after embedding, after encoder, after classifier
# then train a little and report test accuracy
torch.manual_seed(42)
model = TinyNewsEncoder(len(vocab), pad_id=0)
model(next(iter(train_loader))[0], print_shapes=True)
print("*" * 70)
print("\n\n")
torch.manual_seed(42)
criterion = nn.CrossEntropyLoss()
model = TinyNewsEncoder(len(vocab), pad_id=0)
optimizer = torch.optim.Adam(model.parameters())
EPOCHS = 50
metric = torchmetrics.Accuracy(task="multiclass", num_classes=2)

history = train(model, optimizer, criterion, metric, train_loader, EPOCHS)

# print test accuracy
enc_test_acc = evaluate_tm(model, test_loader, metric).item()
### END CODE HERE ###

print("encoder test acc:", enc_test_acc)


token_ids.shape: torch.Size([32, 64])

token_encoding.shape: torch.Size([32, 64, 32])

position_embeddings.shape: torch.Size([64, 32])

output.shape: torch.Size([32, 2])

**********************************************************************



Epoch 1/50, train loss: 0.7830, train metric: 0.5520
Epoch 2/50, train loss: 0.7317, train metric: 0.4960
Epoch 3/50, train loss: 0.6832, train metric: 0.5520
Epoch 4/50, train loss: 0.6915, train metric: 0.5240
Epoch 5/50, train loss: 0.6804, train metric: 0.5560
Epoch 6/50, train loss: 0.6741, train metric: 0.6440
Epoch 7/50, train loss: 0.6475, train metric: 0.6880
Epoch 8/50, train loss: 0.5995, train metric: 0.7400
Epoch 9/50, train loss: 0.5169, train metric: 0.7720
Epoch 10/50, train loss: 0.4239, train metric: 0.8000
Epoch 11/50, train loss: 0.3481, train metric: 0.8680
Epoch 12/50, train loss: 0.3386, train metric: 0.8640
Epoch 13/50, train loss: 0.2641, train metric: 0.8800
Epoch 14/50, train loss: 0.1933, train metric: 0.9320
Epoch 15

C:\Anaconda\envs\torch-gpu\lib\site-packages\torch\nn\modules\transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


encoder test acc: 0.6700000166893005


<div dir="rtl">

### توضیح بخش ۱ـ د

- شکل تنسورها:
- چرا اینجا causal mask نگذاشتید؟
در رمزنگار ایرادی ندارد توکنهای آینده را ببینیم. اینجا تنها میخواهیم معنی را وابسته به کانتکست دریافت کنیم. قرار نیست پیش بینی کنیم که توکن بعدی چه خواهد بود.
</div>


<div dir="rtl">

## بخش ۱ـ ه. مقایسه و جمع‌بندی

جدول را کامل کنید و به سه سؤال PDF جواب دهید.

</div>


<div dir="rtl">

### جدول مقایسه پروژه ۱

| مدل | دقت آزمون | این مدل چه اطلاعاتی می‌بیند؟ |
|---|---|---|
| BoW | 0.85 | Freq. of tokens |
| میانگین embedding | 0.81 | Embeded tokens |
| Encoder | 0.61 | Token's Embeding, Position and Attention |

### سؤال‌ها

1. چرا بدون اطلاعات موقعیت، توجه ترتیب «اول / بعدی» را نمی‌فهمد؟
2. LayerNorm روی چه چیزی اعمال می‌شود؟
3. اگر به‌جای Encoder از RNN یک‌لایه استفاده می‌کردید، گلوگاه اصلی چه بود؟

**پاسخ:**

1.چون در توکنایز کردن جایگاه توکن لحاظ نمیشود.
2.روی خروجی لایههای اتنشن و فید فوروار
3.محدودیت در اندازه پنجره و انفجار یا محو گرادیان

</div>


<div dir="rtl">

# پروژه ۲. قلم نمایشنامه‌نویس: مدل زبانی کوچک

**هدف:** از متن خام، توکن بعدی را یاد بگیرید و بعد از یک پیشوند چند توکن تولید کنید.

**داده:** `Tiny Shakespeare` — فقط حدود ۱۲۰۰۰ تا ۲۰۰۰۰ نویسهٔ اول، نه کل فایل.

</div>


<div dir="rtl">

## بخش ۲ـ الف. متن، نرمال‌سازی و دو نگاه توکنایزری

باید انجام دهید:
- برش کوچک فایل را بخوانید.
- نرمال‌سازی سبک؛ تصمیم‌تان را بعداً توضیح دهید.
- همان دو خط را با `bert-base-uncased` و `gpt2` توکنایز کنید.
- برای مدل خودتان واژه‌نامه بسازید. توصیه: سطح **نویسه**.

</div>


In [13]:
from pathlib import Path
import urllib.request
import transformers

In [14]:
def tokenize_shakespeare_text(text):
    parts = re.findall(
        r'(?ms)^[^:\n]+:\s*\n.*?(?=^[^:\n]+:|\Z)',
        text
    )

    parts = [part.strip() for part in parts]
    return parts

In [15]:
SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)
MAX_CHARS = 16000

def download_shakespeare_text(url=SHAKESPEARE_URL):
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path.read_text()

raw_text = download_shakespeare_text()[:MAX_CHARS]
assert len(raw_text) == MAX_CHARS

text = tokenize_shakespeare_text(raw_text)
text[:3]

['First Citizen:\nBefore we proceed any further, hear me speak.',
 'All:\nSpeak, speak.',
 'First Citizen:\nYou are all resolved rather to die than to famish?']

In [29]:
vocab = sorted(set(char for txt in text for char in txt.lower()))
print(f"vocab: {''.join(vocab)}")

print("chars used:", None)
print(text[:4] if text else None)
print(raw_text[:400])


vocab: 
 !',-.:;?abcdefghijklmnopqrstuvwxyz
chars used: None
['First Citizen:\nBefore we proceed any further, hear me speak.', 'All:\nSpeak, speak.', 'First Citizen:\nYou are all resolved rather to die than to famish?', 'All:\nResolved. resolved.']
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it 


In [18]:
gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

In [19]:
### START CODE HERE ###
# tokenize sample_lines with bert-base-uncased and gpt2
# print tokens, ids, and number of ids for each tokenizer
gpt2_encoding = gpt2_tokenizer(text[:2], truncation=True, padding=True,
                               max_length=500, return_tensors="pt")
bert_ecndoging = bert_tokenizer(text[:2], truncation=True, padding=True,
                               max_length=500, return_tensors="pt")

print(f"gpt2_encoding.keys():\n{gpt2_encoding.keys()}\n")
for i, ids in enumerate(gpt2_encoding["input_ids"]):
    tokens = gpt2_tokenizer.convert_ids_to_tokens(ids)
    print(f"Tokens: {' '.join(tokens)}")
    
print("GPT-2 vocabulary size:", gpt2_tokenizer.vocab_size)
gp2_text_unique_ids = set()
for row in gpt2_encoding["input_ids"]:
    for col in row:
        gp2_text_unique_ids.add(col.item())

print(f"gp2_text_unique_ids: {len(gp2_text_unique_ids)}")
print("\n" * 3)

print(f"bert_ecndoging.keys():\n{bert_ecndoging.keys()}\n")
for i, ids in enumerate(bert_ecndoging['input_ids']):
    print("line: ", i)
    print(" ".join(bert_tokenizer.convert_ids_to_tokens(ids)))

print("BERT vocabulary size:", bert_tokenizer.vocab_size)

bert_text_unique_ids = set()
for row in bert_ecndoging["input_ids"]:
    for col in row:
        bert_text_unique_ids.add(col.item())

print(f"bert_text_unique_ids: {len(bert_text_unique_ids)}")

### END CODE HERE ###


gpt2_encoding.keys():
KeysView({'input_ids': tensor([[ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
          3285,   502,  2740,    13],
        [ 3237,    25,   198,  5248,   461,    11,  2740,    13, 50256, 50256,
         50256, 50256, 50256, 50256]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]])})

Tokens: First ĠCitizen : Ċ Before Ġwe Ġproceed Ġany Ġfurther , Ġhear Ġme Ġspeak .
Tokens: All : Ċ Spe ak , Ġspeak . <|endoftext|> <|endoftext|> <|endoftext|> <|endoftext|> <|endoftext|> <|endoftext|>
GPT-2 vocabulary size: 50257
gp2_text_unique_ids: 18




bert_ecndoging.keys():
KeysView({'input_ids': tensor([[  101,  2034,  6926,  1024,  2077,  2057, 10838,  2151,  2582,  1010,
          2963,  2033,  3713,  1012,   102],
        [  101,  2035,  1024,  3713,  1010,  3713,  1012,   102,     0,     0,
             0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0

In [32]:
### START CODE HERE ###
# build a character-level vocab from `text`
# stoi / itos / stream of ids
itos = {idx: token for idx, token in enumerate(vocab)}
stoi = {token: idx for idx, token in enumerate(vocab)}

print("=" * 70)
print(f"stoi: {stoi}")
print("=" * 70)
print(f"itos: {itos}")
print("=" * 70)

stoi: {'\n': 0, ' ': 1, '!': 2, "'": 3, ',': 4, '-': 5, '.': 6, ':': 7, ';': 8, '?': 9, 'a': 10, 'b': 11, 'c': 12, 'd': 13, 'e': 14, 'f': 15, 'g': 16, 'h': 17, 'i': 18, 'j': 19, 'k': 20, 'l': 21, 'm': 22, 'n': 23, 'o': 24, 'p': 25, 'q': 26, 'r': 27, 's': 28, 't': 29, 'u': 30, 'v': 31, 'w': 32, 'x': 33, 'y': 34, 'z': 35}
itos: {0: '\n', 1: ' ', 2: '!', 3: "'", 4: ',', 5: '-', 6: '.', 7: ':', 8: ';', 9: '?', 10: 'a', 11: 'b', 12: 'c', 13: 'd', 14: 'e', 15: 'f', 16: 'g', 17: 'h', 18: 'i', 19: 'j', 20: 'k', 21: 'l', 22: 'm', 23: 'n', 24: 'o', 25: 'p', 26: 'q', 27: 'r', 28: 's', 29: 't', 30: 'u', 31: 'v', 32: 'w', 33: 'x', 34: 'y', 35: 'z'}


In [34]:
def encode_text(text):
    return torch.tensor([stoi.get(token, '<unk>') for token in text.lower()])

def decode_text(char_ids):
    return "".join([itos[char_id.item()] for char_id in char_ids])

sample_text = "Hello World!"
encoded_text = encode_text(sample_text)
print("=" * 70)
print(f"encoded_text: {encoded_text}")
print("=" * 70)
decoded_text = decode_text(encoded_text)
print(f"decoded_text: {decoded_text}")
print("=" * 70)

encoded_text: tensor([17, 14, 21, 21, 24,  1, 32, 24, 27, 21, 13,  2])
decoded_text: hello world!


In [36]:
stream_ids = [encode_text(char) for txt in text for char in txt]
### END CODE HERE ###

print("char vocab size:", len(vocab))
print("stream length:", len(stream_ids))


char vocab size: 36
stream length: 10309


<div dir="rtl">

### توضیح بخش ۲ـ الف
- **چه نرمال‌سازی‌ای کردید و چرا؟**
- *صرفا سعی شده هر بند را جدا کنم. یعنی هر مکالمه را یک دنباله در نظر بگیرم. چون دنباله گویا مکالمه و متنی است که بازیگر بایستی در نمایش اجرا کند و نخواستم این ساختار را برهم بزنم*
- **BERT و GPT-2 روی همان دو خط چه فرقی داشتند؟**
- *تعداد توکنها متفاوت بود. در جی پی تی اسپیک با اس بزرگ را شکسته بود. تگها اضافی در برت برای طبقه‌بندی کلاس وجود دارد. چون برت مناسب طبقه‌بندی است و تگ سی ال اس مناسب این امر. سایز لغتنامه جی پی تی بزرگتر است.*
- **چرا برای آموزش مدل خودتان واژه‌نامهٔ کوچک ساختید؟**
- *آموزش راحتتر مدل با داشتن لغت نامه کمتر. و اینکه شاید اینجا هدف پیش بینی کاراکتر بعدی باشد واین لغت نامه مناسبتر باشد."
</div>


<div dir="rtl">

## بخش ۲ـ ب. ساخت مثال next-token

باید انجام دهید:
- پنجره‌های طول ثابت، مثلاً ۳۲ یا ۶۴
- ورودی: همه به‌جز آخری ؛ هدف: همه به‌جز اولی
- یک مثال را از ID به متن برگردانید

</div>


In [37]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length
        window = self.encoded_text[idx : end]
        target = self.encoded_text[idx + 1 : end + 1]
        return window, target

to_be_dataset = CharDataset("To be or not to be", window_length=10)
for x, y in to_be_dataset:
    print(f"x={x}, y={y}")
    print(f"    decoded: x={decode_text(x)!r}, y={decode_text(y)!r}")

x=tensor([29, 24,  1, 11, 14,  1, 24, 27,  1, 23]), y=tensor([24,  1, 11, 14,  1, 24, 27,  1, 23, 24])
    decoded: x='to be or n', y='o be or no'
x=tensor([24,  1, 11, 14,  1, 24, 27,  1, 23, 24]), y=tensor([ 1, 11, 14,  1, 24, 27,  1, 23, 24, 29])
    decoded: x='o be or no', y=' be or not'
x=tensor([ 1, 11, 14,  1, 24, 27,  1, 23, 24, 29]), y=tensor([11, 14,  1, 24, 27,  1, 23, 24, 29,  1])
    decoded: x=' be or not', y='be or not '
x=tensor([11, 14,  1, 24, 27,  1, 23, 24, 29,  1]), y=tensor([14,  1, 24, 27,  1, 23, 24, 29,  1, 29])
    decoded: x='be or not ', y='e or not t'
x=tensor([14,  1, 24, 27,  1, 23, 24, 29,  1, 29]), y=tensor([ 1, 24, 27,  1, 23, 24, 29,  1, 29, 24])
    decoded: x='e or not t', y=' or not to'
x=tensor([ 1, 24, 27,  1, 23, 24, 29,  1, 29, 24]), y=tensor([24, 27,  1, 23, 24, 29,  1, 29, 24,  1])
    decoded: x=' or not to', y='or not to '
x=tensor([24, 27,  1, 23, 24, 29,  1, 29, 24,  1]), y=tensor([27,  1, 23, 24, 29,  1, 29, 24,  1, 11])
    decoded: x=

In [52]:
SEQ_LEN = 64

batch_size = 512  # reduce if your GPU cannot handle such a large batch size
train_set = CharDataset("".join(text), SEQ_LEN)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
### START CODE HERE ###
x_lm = None  # [num_windows, SEQ_LEN]
y_lm = None  # [num_windows, SEQ_LEN]
### END CODE HERE ###

# print("x_lm:", None)
# print("y_lm:", None)
# print("example input :", None)
# print("example target:", None)

print(f"len(train_set): {len(train_set)}")
x, y = next(iter(train_loader))
print(f"len(x.shape): {x.shape}")
print(f"len(y.shape): {y.shape}")

len(train_set): 10245
len(x.shape): torch.Size([512, 64])
len(y.shape): torch.Size([512, 64])


<div dir="rtl">

### توضیح بخش ۲ـ ب

- **این کار چرا self-supervised است؟**
- *چون از ورودی برچسب ساخته‌ایم.*
- **در مثال شما، مدل باید بعد از توکن اولِ ورودی چه چیزی را پیش‌بینی کند؟**
- *توکن بعدی. یعنی کاراکتر اول را که گرفت بگوید دومی چیست.*
</div>


<div dir="rtl">

## بخش ۲ـ ج. مدل decoder-style و ماسک علّی

باید انجام دهید:
- embedding توکن + embedding موقعیت + Transformer + لایه به اندازهٔ واژه‌نامه
- **ماسک علّی**
- چاپ ماسک طول ۸
- عبور یک batch و چاپ شکل logits: `[batch, length, vocab]`

</div>


In [61]:
class TinyCharLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=32,
        nhead=4,
        num_layers=2,
        max_len=128,
        dropout=0.1,
    ):
        super().__init__()

        self.token_embed = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_embed = nn.Embedding(
            max_len,
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.output = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, token_ids):

        B, T = token_ids.shape

        if T > self.pos_embed.num_embeddings:
            raise ValueError(
                f"Sequence length ({T}) exceeds "
                f"max_len ({self.pos_embed.num_embeddings})"
            )

        # Token embeddings
        token_embeddings = self.token_embed(token_ids)

        # Position embeddings
        positions = torch.arange(
            T,
            device=token_ids.device
        )

        position_embeddings = self.pos_embed(positions)

        x = token_embeddings + position_embeddings

        # Causal mask
        causal_mask = torch.triu(
            torch.ones(
                T,
                T,
                dtype=torch.bool,
                device=token_ids.device
            ),
            diagonal=1
        )

        # Decoder-only Transformer
        x = self.transformer(
            x,
            mask=causal_mask
        )

        # [B, T, D] -> [B, T, V]
        logits = self.output(x)

        return logits.permute(0, 2, 1)


### START CODE HERE ###
# print an 8x8 causal mask
size = [8] * 2
full_mask = torch.full(size, True)
causal_mask = torch.triu(full_mask, diagonal=1)
print("=" * 70)
print(f"causal_mask:\n{causal_mask}")
print("=" * 70)
# pass a small batch and print logits shape
X, y = next(iter(train_loader))
model = TinyCharLM(len(vocab))
logits = model(X)
print(f"logits: {logits}")
print("=" * 70)
print(f"logits.shape: {logits.shape}")
### END CODE HERE ###


causal_mask:
tensor([[False,  True,  True,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True],
        [False, False, False, False, False, False,  True,  True],
        [False, False, False, False, False, False, False,  True],
        [False, False, False, False, False, False, False, False]])
logits: tensor([[[ 5.7639e-01,  9.4861e-01, -8.1336e-01,  ...,  1.9386e+00,
           2.2270e+00,  1.5876e+00],
         [-4.8424e-01,  6.6187e-02, -4.4917e-01,  ..., -9.6884e-01,
          -1.4398e-01, -2.8320e-01],
         [-1.0152e-01, -8.7905e-01, -1.0298e+00,  ...,  7.6108e-01,
           5.8587e-01, -7.6492e-01],
         ...,
         [ 2.1094e+00,  3.0313e-01, -1.5202e-01,  ..., -1.0711e+00,
           5.2827e-01, -7.3619e-02],
         [ 1.972

<div dir="rtl">

### توضیح بخش ۲ـج

- **`True` در ماسک یعنی چه؟**
- *یعنی این توکنها در فاز حساب کردن اتنشن در نظر گرفته نشوند.*
- **Encoder را انتخاب کردید یا Decoder؟ چرا این انتخاب decoder-style است؟**
- *من اینجا از انکودر استفاده کردم. چون لایه دیکودر یک محاسبه کراس اتنشن هم دارد که برای کار ما مناسب نیست. در این مدل تنها دنبال سلف اتنشن هستیم و فیچر دیگری نیاز نیست.*
- **Decoder نسبت به Encoder چه زیرماژول اضافه‌ای دارد؟**
- *در بالا توضیح داده شد.*
</div>


<div dir="rtl">

## بخش ۲ـ د. آموزش کوتاه و تولید متن

باید انجام دهید:
- چند ده step با CrossEntropyLoss
- تابع تولید: پیشوند → توکن بعدی → اضافه کردن → تکرار
- شروع از `ROMEO:` یا `First Citizen:`
- حدود ۸۰ تا ۱۲۰ نویسه چاپ کنید

</div>


In [67]:
### START CODE HERE ###
# train a little; print loss every few steps
torch.manual_seed(42)
model = TinyCharLM(len(vocab))
EPOCHS = 50
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=len(vocab))
history = train(model, optimizer, xentropy, accuracy, train_loader, EPOCHS)
### END CODE HERE ###


Epoch 1/50, train loss: 3.3172, train metric: 0.1165
Epoch 2/50, train loss: 2.7845, train metric: 0.2132
Epoch 3/50, train loss: 2.6388, train metric: 0.2452
Epoch 4/50, train loss: 2.5611, train metric: 0.2559
Epoch 5/50, train loss: 2.5190, train metric: 0.2625
Epoch 6/50, train loss: 2.4843, train metric: 0.2666
Epoch 7/50, train loss: 2.4589, train metric: 0.2705
Epoch 8/50, train loss: 2.4411, train metric: 0.2769
Epoch 9/50, train loss: 2.4225, train metric: 0.2799
Epoch 10/50, train loss: 2.4021, train metric: 0.2836
Epoch 11/50, train loss: 2.3893, train metric: 0.2897
Epoch 12/50, train loss: 2.3768, train metric: 0.2929
Epoch 13/50, train loss: 2.3523, train metric: 0.2977
Epoch 14/50, train loss: 2.3500, train metric: 0.2998
Epoch 15/50, train loss: 2.3301, train metric: 0.3056
Epoch 16/50, train loss: 2.3252, train metric: 0.3080
Epoch 17/50, train loss: 2.3072, train metric: 0.3117
Epoch 18/50, train loss: 2.2935, train metric: 0.3171
Epoch 19/50, train loss: 2.2842, trai

In [68]:
def generate(model, prefix: str, max_new_tokens: int = 120) -> str:
    ### START CODE HERE ###
    with torch.no_grad():
        for _ in range(max_new_tokens):
            encoded_text = encode_text(prefix).unsqueeze(dim=0)
            Y_logits = model(encoded_text)
            predicted_char_id = Y_logits[0, :, -1].argmax().item()
            prefix += itos[predicted_char_id]
    return prefix
    ### END CODE HERE ###


print(generate(model, "ROMEO:"))


ROMEO:
whe heave the ben theave athe poustound the prome ther mareanouthensea coffoutoustheaneanoneaneatheno' stoune anounenth


<div dir="rtl">

### توضیح بخش ۲ـ د
- **loss چقدر کم شد؟**
- *1.4577*
- **متن تولیدی چه الگویی را تا حدی یاد گرفته و چه چیزی را بلد نیست؟**
- hea مدام تکرار شده.
</div>


<div dir="rtl">

## بخش ۲ـ ه. یک آزمایش کوچک و جمع‌بندی

فقط **یکی** از سه آزمایش PDF را انجام دهید. بعد به دو سؤال آخر جواب دهید.

</div>


In [69]:
class TinyCharLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=32,
        nhead=4,
        num_layers=2,
        max_len=128,
        dropout=0.1,
    ):
        super().__init__()

        self.token_embed = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_embed = nn.Embedding(
            max_len,
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.output = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, token_ids):

        B, T = token_ids.shape

        if T > self.pos_embed.num_embeddings:
            raise ValueError(
                f"Sequence length ({T}) exceeds "
                f"max_len ({self.pos_embed.num_embeddings})"
            )

        # Token embeddings
        token_embeddings = self.token_embed(token_ids)

        # Position embeddings
        positions = torch.arange(
            T,
            device=token_ids.device
        )

        position_embeddings = self.pos_embed(positions)

        x = token_embeddings + position_embeddings

        # Decoder-only Transformer
        x = self.transformer(
            x
        )

        # [B, T, D] -> [B, T, V]
        logits = self.output(x)

        return logits.permute(0, 2, 1)

torch.manual_seed(42)
model = TinyCharLM(len(vocab))
EPOCHS = 50
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=len(vocab))
history = train(model, optimizer, xentropy, accuracy, train_loader, EPOCHS)


C:\Anaconda\envs\torch-gpu\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 1/50, train loss: 3.3133, train metric: 0.1165
Epoch 2/50, train loss: 2.7783, train metric: 0.2143
Epoch 3/50, train loss: 2.6315, train metric: 0.2463
Epoch 4/50, train loss: 2.5530, train metric: 0.2575
Epoch 5/50, train loss: 2.5030, train metric: 0.2654
Epoch 6/50, train loss: 2.4670, train metric: 0.2701
Epoch 7/50, train loss: 2.4295, train metric: 0.2779
Epoch 8/50, train loss: 2.4021, train metric: 0.2863
Epoch 9/50, train loss: 2.3454, train metric: 0.3045
Epoch 10/50, train loss: 2.2346, train metric: 0.3428
Epoch 11/50, train loss: 1.9878, train metric: 0.4278
Epoch 12/50, train loss: 1.5317, train metric: 0.5709
Epoch 13/50, train loss: 1.0057, train metric: 0.7234
Epoch 14/50, train loss: 0.5781, train metric: 0.8502
Epoch 15/50, train loss: 0.3888, train metric: 0.9018
Epoch 16/50, train loss: 0.2995, train metric: 0.9253
Epoch 17/50, train loss: 0.2380, train metric: 0.9403
Epoch 18/50, train loss: 0.1928, train metric: 0.9509
Epoch 19/50, train loss: 0.1639, trai

In [70]:
print(generate(model, "ROMEO:"))


ROMEO:omlooeoooooooooo
olorooeo
oooloooooeooooeooo
ooooooouooooooooooooooooooovooooooooooooooooooooooooooooooooooooool
ooooooo


In [ ]:
### START CODE HERE ###
# optional: train briefly without causal mask
# or without positional embedding
# then write what you observed in the markdown cell below
### END CODE HERE ###


<div dir="rtl">

### توضیح بخش ۲ـ ه

کدام آزمایش را انتخاب کردید و چه دیدید؟
دقت به شدت بالا و لاس به شدت پایین آمد.
**سؤال‌ها**

1. چرا پروژهٔ ۱ به ماسک علّی نیاز نداشت و پروژهٔ ۲ نیاز داشت؟
2. چون تسک اولی برای کلاسیفیکشن بود. قرار است متن انکد شده و طبقه بندی شود و نیازی به ماسک نیست. کل جمله باید دیده شود تا معنی وابسته به متن اخذ گردد.
3. فرق pretraining خودنظارتی این پروژه با fine-tune روی برچسب موضوع خبر چیست؟
4. در پروژه آخر چون قرار است کاراکتر بعدی پیش بینی شود، با شیفت زمانی به داده خروجی میرسیم. مدل با دیدن یک پنجره زمانی به دنبال پیدا کردن جواب در پنجره زمانی آینده است. در مدل ابتدایی بایستی محتوا و اطلاعات یا دانش را از داده ورودی بگیریم و مدل را با داشتن برچسب آن داده‌ها اموزش دهیم. در دومی به برچسب نیاز قطعی داریم ولی در دومی برچسب قابل ساختن است.
**پاسخ:**

1.
2.

</div>


<div dir="rtl">

# چک‌لیست قبل از ارسال

- [ ] پروژهٔ ۱ روی حداکثر ۲۵۰+۱۰۰ سند از دو کلاس خواسته‌شده اجرا شده است.
- [ ] BoW، میانگین embedding و Encoder گزارش شده‌اند.
- [ ] دو سند اشتباه BoW و یک جفت جملهٔ دلخواه آمده است.
- [ ] چند جفت skip-gram ساخته شده است.
- [ ] پروژهٔ ۲ روی برش کوچک Tiny Shakespeare است.
- [ ] BERT و GPT-2 فقط برای مشاهدهٔ دو خط استفاده شده‌اند.
- [ ] یک مثال ورودی/هدف شیفت‌شده چاپ شده است.
- [ ] ماسک علّی چاپ شده و یک متن کوتاه تولید شده است.
- [ ] همهٔ سلول‌های «توضیح این بخش» پر شده‌اند.

</div>
